# Fine-tuning y evaluación de modelos de lenguaje
## Clasificación de emociones en texto con DistilBERT


---
# Bloque 0. Configuración del entorno

**Instrucciones**

1. Abrir este notebook en **Google Colab**.
2. Ir a `Entorno de ejecución > Cambiar tipo de entorno de ejecución` y seleccionar **GPU T4**.
3. Ejecutar las celdas de esta sección.

**Por qué:** el ajuste fino multiplica millones de parámetros en cada paso. En una GPU, una época con 4.000 frases tarda menos de un minuto; en CPU puede tardar más de 20 minutos. El código funciona en ambos casos, solo cambia el tiempo.

In [ ]:
# Instalación de librerías (en Colab la mayoría ya vienen instaladas)
# Este notebook usa la API de transformers 5.x
#!pip install -q "transformers>=5.0" datasets accelerate scikit-learn

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed,
)
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

SEMILLA = 42
set_seed(SEMILLA)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Versión de transformers:", transformers.__version__)
print("Dispositivo:", DISPOSITIVO)
if DISPOSITIVO == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

**Por qué fijamos una semilla:** el ajuste fino tiene componentes aleatorios (inicialización de la cabeza de clasificación, orden de los lotes, dropout). Fijar la semilla hace que los resultados sean **reproducibles**, condición indispensable para comparar experimentos en el Bloque 7.

---
# Bloque 1. Del Transformer al modelo útil (15 min)

## 1.1 Dos etapas en la vida de un modelo de lenguaje

| | Preentrenamiento | Ajuste fino (fine-tuning) |
|---|---|---|
| **Objetivo** | Aprender cómo funciona el lenguaje | Resolver una tarea concreta |
| **Datos** | Enormes cantidades de texto **sin etiquetas** | Pocos miles de ejemplos **etiquetados** |
| **Tarea** | Predecir palabras ocultas (*masked language modeling*) | Clasificar, extraer, responder... |
| **Costo** | Semanas en cientos de GPU | Minutos u horas en una GPU |
| **Quién lo hace** | Grandes laboratorios | Cualquier equipo de desarrollo |

La idea clave es la **transferencia de aprendizaje**: reutilizamos todo lo que el modelo aprendió sobre el lenguaje y solo le enseñamos la tarea final.

## 1.2 La cabeza de clasificación

En la clase anterior vimos el **encoder** del Transformer: recibe tokens y devuelve un vector contextual por cada token. Para clasificar una frase completa necesitamos **un solo vector** que la represente. Por eso los modelos tipo BERT agregan un token especial al inicio, `[CLS]`, cuyo vector final resume la frase.

Sobre ese vector se coloca una **cabeza de clasificación**: una pequeña red densa que transforma el vector de 768 dimensiones en 6 valores (uno por emoción), llamados **logits**. La función softmax convierte los logits en probabilidades:

$$P(y = k \mid x) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

donde $z_k$ es el logit de la clase $k$ y $K = 6$ es el número de emociones.

```
"I feel so happy today"
        |
   Tokenizador  ->  [CLS] i feel so happy today [SEP]
        |
   Encoder DistilBERT (6 capas, preentrenado)
        |
   Vector de [CLS] (768 dimensiones)
        |
   Cabeza de clasificación (NUEVA, pesos aleatorios)
        |
   6 logits -> softmax -> probabilidades por emoción
```

## 1.3 El problema: un modelo que aún no sabe la tarea

Vamos a cargar DistilBERT con una cabeza de clasificación de 6 clases **sin entrenarla** y le pediremos clasificar frases obvias.

In [ ]:
MODELO_BASE = "distilbert-base-uncased"

# Etiquetas del dataset dair-ai/emotion, en el mismo orden que usa el dataset
ETIQUETAS_EN = ["sadness", "joy", "love", "anger", "fear", "surprise"]
ETIQUETAS_ES = ["tristeza", "alegría", "amor", "ira", "miedo", "sorpresa"]

id2label = {i: e for i, e in enumerate(ETIQUETAS_ES)}
label2id = {e: i for i, e in enumerate(ETIQUETAS_ES)}

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
modelo_sin_ajuste = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=len(ETIQUETAS_ES),
    id2label=id2label,
    label2id=label2id,
).to(DISPOSITIVO)

**Observe la advertencia** que aparece al cargar el modelo: indica que algunos pesos (`classifier`, `pre_classifier`) **fueron inicializados aleatoriamente**. Esa es exactamente la cabeza de clasificación nueva. La librería nos está avisando que el modelo debe entrenarse antes de usarse.

In [ ]:
def predecir(modelo, frases, top=2):
    # Devuelve las 'top' emociones más probables para cada frase
    modelo.eval()
    entradas = tokenizer(frases, padding=True, truncation=True,
                         max_length=64, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        logits = modelo(**entradas).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    filas = []
    for frase, p in zip(frases, probs):
        orden = np.argsort(p)[::-1][:top]
        fila = {"frase": frase}
        for r, idx in enumerate(orden, start=1):
            fila[f"emoción {r}"] = modelo.config.id2label[int(idx)]
            fila[f"prob {r}"] = round(float(p[idx]), 3)
        filas.append(fila)
    return pd.DataFrame(filas)

FRASES_DEMO = [
    "I feel so happy and grateful today",
    "I am terrified of what might happen tomorrow",
    "I miss my grandmother so much, I can't stop crying",
    "I am furious, they lied to me again",
    "I love spending time with you",
    "I can't believe it, I won the lottery!",
]

predecir(modelo_sin_ajuste, FRASES_DEMO)

**Pregunta para la clase:** ¿las probabilidades son razonables?

Las probabilidades rondan $1/6 \approx 0{,}17$ para todas las clases y las predicciones no tienen relación con el significado de la frase. El encoder sí "entiende" el texto, pero la cabeza de clasificación todavía no sabe qué emoción corresponde a cada representación.

**Conclusión del bloque:** necesitamos **ajustar** el modelo con ejemplos etiquetados. Ese es el objetivo de esta clase.

In [ ]:
# Liberamos memoria: este modelo solo servía para la demostración
del modelo_sin_ajuste
if DISPOSITIVO == "cuda":
    torch.cuda.empty_cache()

---
# Bloque 2. Anatomía del modelo y sus parámetros (25 min)

**Instrucciones:** ejecute cada celda y relacione lo que ve con la arquitectura Transformer de la clase anterior.

**Por qué:** antes de entrenar un modelo hay que saber **qué** se va a entrenar. El número de parámetros determina el costo en memoria y tiempo, y saber dónde está cada capa permite decidir qué partes congelar.

In [ ]:
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=len(ETIQUETAS_ES),
    id2label=id2label,
    label2id=label2id,
).to(DISPOSITIVO)

print(modelo)

### Lectura guiada de la arquitectura

| Componente | Qué es | Relación con la clase anterior |
|---|---|---|
| `embeddings.word_embeddings` | Tabla de 30.522 tokens x 768 dimensiones | Vectores iniciales de cada token |
| `embeddings.position_embeddings` | 512 posiciones x 768 dimensiones | Codificación posicional |
| `transformer.layer[0..5]` | 6 bloques Transformer | Atención multi-cabeza + red feed-forward |
| `attention.q_lin`, `k_lin`, `v_lin` | Proyecciones lineales | Matrices $W_Q$, $W_K$, $W_V$ |
| `ffn.lin1`, `ffn.lin2` | Red feed-forward (768 -> 3072 -> 768) | Capa posición a posición |
| `pre_classifier`, `classifier` | **Cabeza nueva** (768 -> 768 -> 6) | Lo que agregamos para la tarea |

DistilBERT es una versión **destilada** de BERT: conserva cerca del 97% de su desempeño con 6 capas en lugar de 12, por eso es ideal para clase.

In [ ]:
def contar_parametros(modelo):
    total = sum(p.numel() for p in modelo.parameters())
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    return total, entrenables

total, entrenables = contar_parametros(modelo)
print(f"Parámetros totales:     {total:>12,}")
print(f"Parámetros entrenables: {entrenables:>12,}")

# Parámetros por componente
componentes = {
    "Embeddings": modelo.base_model.embeddings,
    "Capas Transformer (6)": modelo.base_model.transformer,
    "Cabeza de clasificación": torch.nn.ModuleList([modelo.pre_classifier, modelo.classifier]),
}
resumen = pd.DataFrame([
    {"componente": n, "parámetros": sum(p.numel() for p in m.parameters())}
    for n, m in componentes.items()
])
resumen["porcentaje"] = (100 * resumen["parámetros"] / total).round(2)
resumen

**Pregunta para la clase:** la cabeza de clasificación representa menos del 1% del modelo. ¿Qué implica entrenar solo esa parte frente a entrenar todo el modelo?

## 2.1 El tokenizador

El modelo no lee texto, lee números. El tokenizador divide la frase en **subpalabras** (algoritmo WordPiece) y las convierte en identificadores.

In [ ]:
frase = "I am unbelievably happy with the results"
codificado = tokenizer(frase)

print("Tokens:        ", tokenizer.convert_ids_to_tokens(codificado["input_ids"]))
print("input_ids:     ", codificado["input_ids"])
print("attention_mask:", codificado["attention_mask"])

Observe:

- `[CLS]` (id 101) al inicio y `[SEP]` (id 102) al final se agregan automáticamente.
- Palabras poco frecuentes se dividen en subpalabras con el prefijo `##` (por ejemplo `unbelievably`). Así el vocabulario se mantiene en unos 30.000 tokens sin palabras desconocidas.
- `attention_mask` vale 1 en tokens reales y 0 en tokens de relleno (*padding*). Le indica al mecanismo de atención qué posiciones ignorar.

## 2.2 Tres estrategias de ajuste

| Estrategia | Qué se entrena | Ventaja | Desventaja |
|---|---|---|---|
| **Extracción de características** (encoder congelado) | Solo la cabeza (~0,6 M parámetros) | Muy rápido, poca memoria | El encoder no se adapta al dominio |
| **Ajuste fino completo** | Todo el modelo (~67 M parámetros) | Mejor desempeño | Más costo, riesgo de olvido catastrófico |
| **PEFT / LoRA** | Pequeñas matrices añadidas a las capas de atención | Casi el desempeño del ajuste completo con ~1% de los parámetros | Más complejo de configurar |

En esta clase compararemos las dos primeras. LoRA es la estrategia estándar para modelos grandes (LLM de miles de millones de parámetros), donde el ajuste completo es inviable.

In [ ]:
def congelar_encoder(modelo):
    # Congela todo el encoder: solo la cabeza de clasificación queda entrenable
    for p in modelo.base_model.parameters():
        p.requires_grad = False
    return modelo

# Demostración con una copia temporal
demo = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=6)
demo = congelar_encoder(demo)
t, e = contar_parametros(demo)
print(f"Encoder congelado -> entrenables: {e:,} de {t:,} ({100*e/t:.2f} %)")
del demo

---
# Bloque 3. Preparación de datos (20 min)

**Dataset:** [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion), frases en inglés tomadas de Twitter y etiquetadas con 6 emociones básicas.

**Por qué este dataset:** está limpio, documentado y presenta **desbalance de clases**, situación muy frecuente en proyectos reales y que nos obligará a elegir bien la métrica de evaluación.

In [ ]:
dataset = load_dataset("dair-ai/emotion")
dataset

In [ ]:
# Verificamos que el orden de las etiquetas coincide con el que definimos
print(dataset["train"].features["label"].names)
assert dataset["train"].features["label"].names == ETIQUETAS_EN

In [ ]:
df_train = dataset["train"].to_pandas()
df_train["emoción"] = df_train["label"].map(id2label)
df_train.sample(8, random_state=SEMILLA)

In [ ]:
conteo = df_train["emoción"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
conteo.plot(kind="barh", ax=ax, color="#4C72B0")
ax.invert_yaxis()
ax.set_title("Distribución de emociones en el conjunto de entrenamiento")
ax.set_xlabel("Número de frases")
for i, v in enumerate(conteo.values):
    ax.text(v + 50, i, f"{v} ({100*v/len(df_train):.1f}%)", va="center")
plt.tight_layout()
plt.show()

**Pregunta para la clase:** si un modelo respondiera siempre "alegría", ¿qué accuracy obtendría? (Aproximadamente un tercio). Por eso la accuracy sola es engañosa con clases desbalanceadas.

## 3.1 Longitud de las frases

Necesitamos decidir el `max_length` del tokenizador: frases más largas se truncan.

In [ ]:
longitudes = [len(tokenizer(t)["input_ids"]) for t in df_train["text"].sample(2000, random_state=SEMILLA)]
print(f"Percentil 95 de longitud en tokens: {np.percentile(longitudes, 95):.0f}")
print(f"Máximo: {max(longitudes)}")

plt.figure(figsize=(8, 3))
plt.hist(longitudes, bins=40, color="#55A868")
plt.axvline(64, color="red", linestyle="--", label="max_length = 64")
plt.xlabel("Tokens por frase"); plt.ylabel("Frecuencia"); plt.legend()
plt.show()

Con `max_length = 64` cubrimos prácticamente todas las frases. Como el costo de la atención crece con el **cuadrado** de la longitud, una longitud corta acelera mucho el entrenamiento.

## 3.2 Submuestra y particiones

**Instrucciones:** usaremos 4.000 frases de entrenamiento, el conjunto de validación completo para monitorear el entrenamiento y el conjunto de prueba completo **solo al final**.

**Por qué:** 4.000 ejemplos bastan para un buen resultado en pocos minutos y permiten repetir experimentos en clase. El conjunto de prueba no se usa para tomar decisiones; si se usara, la evaluación final estaría sesgada.

In [ ]:
N_TRAIN = 16000   # puede subir a 16000 (dataset completo) si dispone de tiempo

train_ds = dataset["train"].shuffle(seed=SEMILLA).select(range(N_TRAIN))
val_ds = dataset["validation"]
test_ds = dataset["test"]

print(f"Entrenamiento: {len(train_ds):,} | Validación: {len(val_ds):,} | Prueba: {len(test_ds):,}")

In [ ]:
MAX_LEN = 64

def tokenizar(lote):
    return tokenizer(lote["text"], truncation=True, max_length=MAX_LEN)

train_tok = train_ds.map(tokenizar, batched=True)
val_tok = val_ds.map(tokenizar, batched=True)
test_tok = test_ds.map(tokenizar, batched=True)

# Padding dinámico: cada lote se rellena solo hasta la frase más larga del lote
collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_tok

**Por qué padding dinámico:** si rellenáramos todas las frases a 64 tokens, la mayoría de los cálculos se harían sobre ceros. `DataCollatorWithPadding` rellena cada lote solo hasta la frase más larga de ese lote, lo que ahorra tiempo sin cambiar el resultado.

---
# Bloque 4. Ajuste fino (40 min)

## 4.1 Qué ocurre durante el entrenamiento

En cada paso:

1. Se toma un **lote** de frases y se calculan las predicciones (paso hacia adelante).
2. Se mide el error con la **entropía cruzada**:
$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log P(y_i \mid x_i)$$
   donde $y_i$ es la emoción correcta de la frase $x_i$. Si el modelo asigna probabilidad alta a la clase correcta, la pérdida es baja.
3. Se calculan los gradientes (retropropagación) y el optimizador **AdamW** actualiza los pesos:
$$\theta \leftarrow \theta - \eta \cdot \text{paso}_{\text{AdamW}}(\nabla_\theta \mathcal{L})$$
   donde $\eta$ es la **tasa de aprendizaje** (*learning rate*).

## 4.2 Hiperparámetros

| Hiperparámetro | Valor | Por qué |
|---|---|---|
| `learning_rate` | 2e-5 | Muy bajo a propósito: pasos grandes destruyen lo aprendido en el preentrenamiento (**olvido catastrófico**) |
| `per_device_train_batch_size` | 16 | Frases procesadas por paso; equilibrio entre memoria de GPU y estabilidad del gradiente |
| `num_train_epochs` | 3 | Veces que el modelo recorre todo el conjunto; con 2-4 suele bastar en ajuste fino |
| `weight_decay` | 0.01 | Regularización: penaliza pesos grandes para reducir sobreajuste |
| `warmup_steps` | 0.1 | Un valor entre 0 y 1 se interpreta como proporción: el 10% inicial de los pasos sube la tasa de aprendizaje gradualmente, evitando cambios bruscos cuando la cabeza aún es aleatoria |

**Nota técnica (transformers 5.x):** muchos tutoriales antiguos usan argumentos que ya no existen y producen error:

- `evaluation_strategy` ahora se llama `eval_strategy`.
- `warmup_ratio` se eliminó: se usa `warmup_steps` con un valor decimal entre 0 y 1.
- `Trainer(tokenizer=...)` ahora es `Trainer(processing_class=...)`.

In [ ]:
def calcular_metricas(eval_pred):
    logits, etiquetas = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(etiquetas, preds),
        "f1_macro": f1_score(etiquetas, preds, average="macro"),
    }

args = TrainingArguments(
    output_dir="salida_emociones",

    # Parámetros de entrenamiento
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_steps=0.1,

    # Evaluar y guardar al final de cada época
    eval_strategy="epoch",
    save_strategy="epoch",

    # Recuperar el modelo con mejor F1-macro en validación
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,

    # Registro y configuración
    logging_steps=25,
    fp16=(DISPOSITIVO == "cuda"),
    report_to="none",
    seed=SEMILLA,
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=calcular_metricas,
)

pasos_por_epoca = int(np.ceil(N_TRAIN / args.per_device_train_batch_size))
print(f"Pasos por época: {pasos_por_epoca} | Pasos totales: {pasos_por_epoca * int(args.num_train_epochs)}")

print("Mejor F1-macro:", trainer.state.best_metric)
print("Carpeta del mejor modelo:", trainer.state.best_model_checkpoint)

**Instrucciones:** ejecute la siguiente celda. Mientras entrena (1-3 minutos en GPU), observe cómo baja la pérdida y discuta con su grupo: ¿qué pasaría con una tasa de aprendizaje 100 veces mayor?

In [ ]:
resultado = trainer.train()
print(resultado)

## 4.3 Curvas de aprendizaje

**Por qué:** las curvas son el diagnóstico principal del entrenamiento.

- Si la pérdida de **entrenamiento** baja y la de **validación** también: el modelo aprende y generaliza.
- Si la de entrenamiento sigue bajando pero la de validación **sube**: **sobreajuste**, el modelo memoriza en lugar de aprender.

In [ ]:
historial = pd.DataFrame(trainer.state.log_history)
perd_train = historial.dropna(subset=["loss"])[["epoch", "loss"]]
perd_val = historial.dropna(subset=["eval_loss"])[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(perd_train["epoch"], perd_train["loss"], label="Entrenamiento", alpha=0.8)
axes[0].plot(perd_val["epoch"], perd_val["eval_loss"], "o-", label="Validación", linewidth=2)
axes[0].set_title("Pérdida (entropía cruzada)"); axes[0].set_xlabel("Época"); axes[0].legend()

axes[1].plot(perd_val["epoch"], perd_val["eval_accuracy"], "o-", label="Accuracy")
axes[1].plot(perd_val["epoch"], perd_val["eval_f1_macro"], "s-", label="F1-macro")
axes[1].set_title("Métricas en validación"); axes[1].set_xlabel("Época"); axes[1].set_ylim(0, 1); axes[1].legend()
plt.tight_layout(); plt.show()

perd_val.round(4)

## 4.4 Guardar el modelo ajustado

**Por qué:** un modelo ajustado es un artefacto reutilizable. Se guardan los pesos y el tokenizador juntos, porque el modelo solo funciona con el mismo tokenizador con el que fue entrenado.

In [ ]:
RUTA_MODELO = "modelo_emociones_distilbert"
trainer.save_model(RUTA_MODELO)   # guarda pesos, configuración y tokenizador

# Opcional (Colab): copiar a Google Drive para conservarlo
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r modelo_emociones_distilbert /content/drive/MyDrive/

---
# Receso (10 min)
---

# Bloque 5. Probar el modelo (20 min)

**Probar no es lo mismo que evaluar.** Probar es interactuar con el modelo para formar una intuición cualitativa de cómo se comporta. Evaluar (Bloque 6) es medir de forma sistemática con datos que el modelo nunca vio.

## 5.1 Las mismas frases del inicio

In [ ]:
predecir(trainer.model, FRASES_DEMO)

**Compare** con la tabla del Bloque 1: con el mismo encoder y la misma arquitectura, unos minutos de ajuste fino cambian completamente el comportamiento.

## 5.2 Uso como pipeline

En producción se usa la interfaz `pipeline`, que encapsula tokenización, inferencia y softmax.

In [ ]:
clasificador = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0 if DISPOSITIVO == "cuda" else -1,
)
clasificador("I just got accepted into my dream university")

## 5.3 Casos difíciles

**Instrucciones:** cada grupo agrega al menos 3 frases propias a la lista de abajo, intentando "engañar" al modelo. Anote cuáles falla y proponga una explicación.

In [ ]:
FRASES_DIFICILES = [
    # Negaciones
    "I am not sad at all",
    "I don't hate you",
    # Sarcasmo
    "Oh great, another Monday. I'm just thrilled",
    # Ambigüedad
    "I can't believe what just happened",
    # Emoción mixta
    "I'm happy for her but I will miss her terribly",
    # Frases de los estudiantes:
    # "...",
]
predecir(trainer.model, FRASES_DIFICILES, top=3)

## 5.4 ¿Y en español?

In [ ]:
FRASES_ES = [
    "Estoy muy feliz con mis resultados",
    "Tengo mucho miedo de lo que pueda pasar",
    "Estoy furioso, me volvieron a mentir",
    "Te quiero muchísimo",
]
predecir(trainer.model, FRASES_ES, top=3)

**Discusión:** el modelo falla o acierta por casualidad. Razones:

1. `distilbert-base-uncased` fue **preentrenado solo en inglés**: su vocabulario parte las palabras en español en fragmentos sin significado.
2. Los datos de ajuste fino también están en inglés.

**¿Qué necesitaríamos para el español?**

- Un modelo base en español, por ejemplo `dccuchile/bert-base-spanish-wwm-cased` (BETO) o `PlanTL-GOB-ES/roberta-base-bne`, o uno multilingüe como `xlm-roberta-base`.
- Datos etiquetados en español.

**Lección general:** el ajuste fino adapta el modelo a una **tarea**, pero no le da conocimientos que no adquirió en el preentrenamiento.

---
# Bloque 6. Evaluación formal (35 min)

**Instrucciones:** a partir de aquí usamos el **conjunto de prueba** (2.000 frases que el modelo nunca vio ni se usaron para decidir nada).

## 6.1 Métricas

Para cada clase $k$:

$$\text{Precisión}_k = \frac{VP_k}{VP_k + FP_k} \qquad \text{Recall}_k = \frac{VP_k}{VP_k + FN_k} \qquad F1_k = 2\cdot\frac{\text{Precisión}_k \cdot \text{Recall}_k}{\text{Precisión}_k + \text{Recall}_k}$$

- **Precisión:** de las veces que el modelo dijo "miedo", ¿cuántas era realmente miedo?
- **Recall (exhaustividad):** de todas las frases de miedo, ¿cuántas detectó?
- **F1-macro:** promedio simple del F1 de las 6 clases:
$$F1_{\text{macro}} = \frac{1}{K}\sum_{k=1}^{K} F1_k$$

**Por qué F1-macro:** da el mismo peso a todas las clases. Una clase minoritaria como "sorpresa" pesa igual que "alegría", así que el modelo no puede obtener buena nota ignorando las clases pequeñas.

In [ ]:
pred_test = trainer.predict(test_tok)
y_true = pred_test.label_ids
y_pred = np.argmax(pred_test.predictions, axis=-1)

print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"F1-macro: {f1_score(y_true, y_pred, average='macro'):.4f}\n")
print(classification_report(y_true, y_pred, target_names=ETIQUETAS_ES, digits=3))

**Pregunta para la clase:** ¿qué clases tienen el F1 más bajo? ¿Coinciden con las clases minoritarias del gráfico del Bloque 3?

## 6.2 Matriz de confusión

**Por qué:** las métricas dicen **cuánto** se equivoca el modelo; la matriz de confusión dice **con qué** se confunde. Está normalizada por fila: cada fila suma 1 y muestra hacia dónde van las frases de cada emoción real.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred,
    display_labels=ETIQUETAS_ES,
    normalize="true",
    values_format=".2f",
    cmap="Blues",
    ax=ax,
)
ax.set_title("Matriz de confusión normalizada (conjunto de prueba)")
ax.set_xlabel("Predicción"); ax.set_ylabel("Emoción real")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

Confusiones típicas en este dataset: **amor con alegría** y **sorpresa con miedo**. Ambas parejas comparten vocabulario y, en algunos casos, incluso a una persona le costaría decidir.

## 6.3 Análisis de errores

**Instrucciones:** revise los errores en los que el modelo estaba **más seguro**. Para cada uno, decida: ¿se equivocó el modelo o la etiqueta original es discutible?

**Por qué:** el análisis de errores es la fuente principal de ideas para mejorar un modelo. A veces revela problemas en los datos que ninguna métrica muestra.

In [ ]:
probs_test = torch.softmax(torch.tensor(pred_test.predictions), dim=-1).numpy()

errores = pd.DataFrame({
    "frase": test_ds["text"],
    "real": [id2label[i] for i in y_true],
    "predicha": [id2label[i] for i in y_pred],
    "confianza": probs_test.max(axis=1).round(3),
})
errores = errores[errores["real"] != errores["predicha"]].sort_values("confianza", ascending=False)

print(f"Total de errores: {len(errores)} de {len(test_ds)}")
pd.set_option("display.max_colwidth", 120)
errores.head(15)

## 6.4 Comparación con líneas base

**Por qué:** un 90% aislado no dice nada. Necesitamos saber cuánto mejor es el ajuste fino frente a alternativas más simples y baratas. Comparamos tres enfoques con **los mismos datos de entrenamiento y prueba**:

1. **TF-IDF + regresión logística:** el enfoque clásico que vimos en la clase de embeddings.
2. **DistilBERT con encoder congelado:** solo se entrena la cabeza.
3. **DistilBERT con ajuste fino completo:** el modelo que acabamos de entrenar.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizador = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = vectorizador.fit_transform(train_ds["text"])
X_test = vectorizador.transform(test_ds["text"])

regresion = LogisticRegression(max_iter=2000, C=5)
regresion.fit(X_train, train_ds["label"])
pred_tfidf = regresion.predict(X_test)

resultados = [{
    "enfoque": "TF-IDF + regresión logística",
    "parámetros entrenables": X_train.shape[1] * 6 + 6,
    "accuracy": accuracy_score(y_true, pred_tfidf),
    "f1_macro": f1_score(y_true, pred_tfidf, average="macro"),
}]
pd.DataFrame(resultados).round(4)

Ahora la segunda línea base. Como solo se entrena la cabeza, usamos una tasa de aprendizaje mayor (1e-3): no hay pesos preentrenados que proteger.

In [ ]:
def entrenar_y_evaluar(nombre, learning_rate=2e-5, epocas=3, congelar=False, mostrar=True):
    # Entrena un modelo nuevo desde DistilBERT preentrenado y lo evalúa en prueba
    set_seed(SEMILLA)
    m = AutoModelForSequenceClassification.from_pretrained(
        MODELO_BASE, num_labels=6, id2label=id2label, label2id=label2id
    )
    if congelar:
        congelar_encoder(m)
    _, n_entrenables = contar_parametros(m)

    a = TrainingArguments(
        output_dir=f"salida_{nombre}",
        learning_rate=learning_rate,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        num_train_epochs=epocas,
        weight_decay=0.01,
        warmup_steps=0.1,
        eval_strategy="no",
        save_strategy="no",
        logging_strategy="no",
        fp16=(DISPOSITIVO == "cuda"),
        report_to="none",
        seed=SEMILLA,
    )
    t = Trainer(model=m, args=a, train_dataset=train_tok,
                data_collator=collator, processing_class=tokenizer)
    t.train()
    p = np.argmax(t.predict(test_tok).predictions, axis=-1)

    fila = {
        "enfoque": nombre,
        "parámetros entrenables": n_entrenables,
        "accuracy": accuracy_score(y_true, p),
        "f1_macro": f1_score(y_true, p, average="macro"),
    }
    del m, t
    if DISPOSITIVO == "cuda":
        torch.cuda.empty_cache()
    if mostrar:
        print(f"{nombre}: F1-macro = {fila['f1_macro']:.4f}")
    return fila

resultados.append(entrenar_y_evaluar("DistilBERT congelado (solo cabeza)",
                                     learning_rate=1e-3, epocas=3, congelar=True))

In [ ]:
_, n_completo = contar_parametros(trainer.model)
resultados.append({
    "enfoque": "DistilBERT ajuste fino completo",
    "parámetros entrenables": n_completo,
    "accuracy": accuracy_score(y_true, y_pred),
    "f1_macro": f1_score(y_true, y_pred, average="macro"),
})

tabla = pd.DataFrame(resultados)
tabla["parámetros entrenables"] = tabla["parámetros entrenables"].map("{:,}".format)
tabla.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(tabla["enfoque"], tabla["f1_macro"], color=["#8C8C8C", "#DD8452", "#4C72B0"])
for i, v in enumerate(tabla["f1_macro"]):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center")
ax.set_xlim(0, 1); ax.set_xlabel("F1-macro en prueba")
ax.set_title("Comparación de enfoques (mismos datos de entrenamiento y prueba)")
ax.invert_yaxis(); plt.tight_layout(); plt.show()

**Preguntas para la clase:**

1. El encoder congelado, ¿supera a TF-IDF? ¿Por qué podría no hacerlo, si DistilBERT "entiende" mejor el lenguaje?
2. ¿Cuánto mejora el ajuste fino completo frente a la mejor línea base? ¿Justifica el costo adicional en parámetros y tiempo?
3. Si tuviera que desplegar este clasificador en un servidor sin GPU, ¿cuál elegiría y por qué?

**Idea clave:** los vectores de `[CLS]` de un modelo preentrenado no están optimizados para emociones. El ajuste fino completo **reorganiza** las representaciones internas para que las emociones queden separadas, y eso no se logra entrenando solo la cabeza.

---
# Bloque 7. Experimento por grupos y cierre (15 min)

**Instrucciones:**

1. Cada grupo ejecuta **un solo** experimento de la tabla, cambiando **una sola** variable respecto a la configuración base (`learning_rate=2e-5`, `epocas=3`, ajuste completo).
2. Anote su F1-macro en la tabla compartida del curso.
3. Prepare una explicación de 1 minuto del resultado.

| Grupo | Variable modificada | Llamada |
|---|---|---|
| 1 | Tasa de aprendizaje alta | `entrenar_y_evaluar("lr_5e-5", learning_rate=5e-5)` |
| 2 | Tasa de aprendizaje muy alta | `entrenar_y_evaluar("lr_1e-4", learning_rate=1e-4)` |
| 3 | Tasa de aprendizaje extrema | `entrenar_y_evaluar("lr_1e-3", learning_rate=1e-3)` |
| 4 | Una sola época | `entrenar_y_evaluar("1_epoca", epocas=1)` |
| 5 | Más épocas | `entrenar_y_evaluar("6_epocas", epocas=6)` |

**Por qué una variable a la vez:** si se cambian dos hiperparámetros a la vez y el resultado mejora, no se puede saber cuál produjo el efecto. Es el mismo principio del diseño experimental.

In [ ]:
# Ejecute SOLO la línea de su grupo
experimento = entrenar_y_evaluar("lr_5e-5", learning_rate=5e-5)
experimento

### Resultados esperados (para la discusión)

- **Tasa alta (1e-3):** el F1 suele colapsar; el modelo puede terminar prediciendo una sola clase. Es el **olvido catastrófico** en acción.
- **Una época:** desempeño algo menor, el modelo aún no converge.
- **Seis épocas:** mejora poco o nada; en validación puede aparecer sobreajuste.

## Cierre: síntesis de la clase

1. El **preentrenamiento** aporta conocimiento general del lenguaje; el **ajuste fino** lo adapta a una tarea con pocos datos.
2. Un modelo preentrenado con cabeza nueva **no sirve** sin ajuste: la cabeza empieza aleatoria.
3. La **tasa de aprendizaje** es el hiperparámetro más delicado del ajuste fino.
4. **Probar** forma intuición; **evaluar** mide. Ambas cosas son necesarias.
5. Con clases desbalanceadas se reporta **F1-macro** y la **matriz de confusión**, no solo accuracy.
6. Un resultado solo tiene sentido **comparado** con líneas base.

## Tarea: clasificador de estilo en español

**Objetivo:** aplicar el proceso completo de esta clase a un problema nuevo, en español.

1. Cada equipo construye un dataset de **al menos 200 frases en español** etiquetadas en dos estilos, por ejemplo **formal / informal** o **irónico / literal**, balanceado entre clases.
2. Divida el dataset en entrenamiento (70%), validación (15%) y prueba (15%).
3. Ajuste un modelo en español (`dccuchile/bert-base-spanish-wwm-cased`) reutilizando el código de este notebook.
4. Compare contra TF-IDF + regresión logística.
5. Entregue el notebook con: curvas de aprendizaje, `classification_report`, matriz de confusión, 10 errores analizados y una conclusión.

---
# Anexo. Cargar el modelo de respaldo

Si el entrenamiento no puede ejecutarse en clase (sin GPU o sin conexión), el docente comparte la carpeta `modelo_emociones_distilbert` generada en el Bloque 4.4. Súbala al entorno (o móntela desde Drive) y ejecute:

In [ ]:
# RUTA_MODELO = "/content/drive/MyDrive/modelo_emociones_distilbert"
modelo_respaldo = AutoModelForSequenceClassification.from_pretrained(RUTA_MODELO).to(DISPOSITIVO)
tokenizer = AutoTokenizer.from_pretrained(RUTA_MODELO)

predecir(modelo_respaldo, FRASES_DEMO)

Con el modelo de respaldo cargado, los Bloques 5 y 6 funcionan igual reemplazando `trainer.model` por `modelo_respaldo`. Para obtener `pred_test` sin `trainer`, cree uno solo para predecir:

```python
trainer = Trainer(model=modelo_respaldo, data_collator=collator, processing_class=tokenizer)
```

---
**Referencias**

- Saravia, E. et al. (2018). *CARER: Contextualized Affect Representations for Emotion Recognition*. EMNLP. (Dataset `dair-ai/emotion`)
- Sanh, V. et al. (2019). *DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter*. arXiv:1910.01108.
- Devlin, J. et al. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.
- Hu, E. et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*. arXiv:2106.09685.
- Tunstall, L., von Werra, L. y Wolf, T. (2022). *Natural Language Processing with Transformers*. O'Reilly.

---
# Tarea: clasificador de estilo en español (formal e informal)

Este bloque es independiente del experimento de emociones. Incluye un conjunto sintético original de frases emparejadas por tema, dos modelos, gráficas, evaluación en prueba y revisión de errores. Cada pareja expresa una idea similar en ambos estilos. La etiqueta **formal** representa un registro cuidado o institucional; **informal**, un registro conversacional. El grado de formalidad depende del contexto y algunas etiquetas admiten discusión.

**Para ejecutarlo:** reinicie el kernel para liberar la memoria de los modelos anteriores y ejecute únicamente las celdas de esta sección en orden. Se necesita conexión la primera vez para descargar BETO y espacio para los checkpoints. Se recomienda GPU; en CPU puede tardar bastante más. El dataset está incorporado en el notebook y se exporta como CSV; no depende de subir un archivo. Los resultados y conclusiones se generan al ejecutar el entrenamiento, no están inventados.

In [ ]:
# Si faltan dependencias, ejecute UNA VEZ esta celda y reinicie el kernel.
# En Colab, PyTorch suele venir instalado. En otros entornos, instálelo según su GPU.
# %pip install -q "transformers>=5,<6" datasets accelerate scikit-learn pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, set_seed)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from IPython.display import display
SEMILLA_ESTILO = 42
set_seed(SEMILLA_ESTILO)
DISPOSITIVO_ESTILO = "cuda" if torch.cuda.is_available() else "cpu"
NOMBRES_ESTILO = ["formal", "informal"]
print("Transformers:", transformers.__version__, "| Dispositivo:", DISPOSITIVO_ESTILO)
if DISPOSITIVO_ESTILO == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Dataset y particiones

Cada fila corresponde a una frase creada para esta práctica. Se agrupan las dos versiones de cada idea y **todo el grupo** se asigna a una sola partición: así el entrenamiento no ve la versión equivalente de una frase de prueba. Se usa el mismo conjunto de entrenamiento y prueba para ambos modelos. Este dataset es pequeño y artificial; sus métricas no garantizan el mismo rendimiento en mensajes reales.

In [ ]:
# Cada pareja contiene dos versiones del mismo tema: formal e informal.
PARES_ESTILO = [['Le agradecería que enviara el informe antes del viernes.', 'Mándame el informe antes del viernes, porfa.'], ['La reunión comenzará a las nueve de la mañana.', 'La reunión arranca a las nueve de la mañana.'], ['No podré asistir a la conferencia por motivos personales.', 'No puedo ir a la charla por unos asuntos míos.'], ['¿Podría indicarme dónde se encuentra la biblioteca?', '¿Me dices dónde queda la biblioteca?'], ['Solicito una prórroga para presentar el proyecto.', '¿Me dan unos días más para entregar el proyecto?'], ['Le comunicamos que su solicitud ha sido aprobada.', 'Te contamos que sí aprobaron tu solicitud.'], ['Agradezco su colaboración durante el proceso.', 'Gracias por echarme una mano con todo esto.'], ['Es necesario revisar los resultados antes de publicarlos.', 'Hay que revisar los resultados antes de subirlos.'], ['Adjunto los documentos requeridos para la matrícula.', 'Te adjunto los papeles que pedían para la matrícula.'], ['Por favor, mantenga su teléfono en modo silencioso.', 'Pon el teléfono en silencio, por favor.'], ['¿Sería posible modificar la fecha de entrega?', '¿Se puede cambiar la fecha de entrega?'], ['Lamento informarle que el producto se encuentra agotado.', 'Qué pena, pero el producto ya se acabó.'], ['El vehículo presenta una falla en el sistema de frenos.', 'El carro tiene un problema con los frenos.'], ['Le solicito que confirme su participación en el evento.', 'Avísame si vas a participar en el evento.'], ['La inscripción estará disponible hasta el lunes.', 'Te puedes inscribir hasta el lunes.'], ['No se permite consumir alimentos dentro del laboratorio.', 'No coman dentro del laboratorio.'], ['Me gustaría conocer su opinión acerca de la propuesta.', 'Quisiera saber qué piensas de la propuesta.'], ['El paquete será entregado en el transcurso de la tarde.', 'El paquete te llega en la tarde.'], ['Ruego disculpar las molestias ocasionadas por la demora.', 'Perdón por todas las molestias de la demora.'], ['La profesora explicó nuevamente el procedimiento.', 'La profe volvió a explicar cómo se hacía.'], ['Le agradecería responder a la mayor brevedad posible.', 'Respóndeme apenas puedas, porfa.'], ['Hemos recibido su mensaje y lo revisaremos oportunamente.', 'Ya vimos tu mensaje y luego lo revisamos.'], ['El acceso permanecerá cerrado durante el mantenimiento.', 'La entrada va a estar cerrada mientras hacen el arreglo.'], ['¿Podría ayudarme a resolver esta inquietud?', '¿Me ayudas con esta duda?'], ['Los participantes deberán presentar un documento de identidad.', 'Todos tienen que llevar su cédula.'], ['Se recomienda guardar una copia de seguridad del archivo.', 'Mejor guarda una copia del archivo por si acaso.'], ['El estudiante obtuvo una calificación sobresaliente.', 'Al chico le fue súper bien en la evaluación.'], ['Le informo que llegaré unos minutos después de lo previsto.', 'Voy a llegar un poquito tarde, te aviso.'], ['Su presencia en la ceremonia será muy apreciada.', 'Nos encantaría que vengas a la ceremonia.'], ['El establecimiento atiende de lunes a sábado.', 'El local abre de lunes a sábado.'], ['Por favor, verifique que sus datos sean correctos.', 'Revisa que tus datos estén bien, porfa.'], ['El examen comprenderá los contenidos de las tres unidades.', 'En el examen entran las tres unidades.'], ['La compañía realizará una actualización del sistema.', 'La empresa va a actualizar el sistema.'], ['Quedo a la espera de sus comentarios sobre el borrador.', 'Me cuentas qué te parece el borrador.'], ['La actividad fue suspendida debido a las condiciones climáticas.', 'Cancelaron la actividad porque el clima estaba feo.'], ['Agradeceríamos que completara la encuesta de satisfacción.', '¿Nos ayudas llenando la encuesta?'], ['Es indispensable respetar los plazos establecidos.', 'Hay que cumplir las fechas que nos dieron.'], ['El personal atenderá sus consultas a partir de mañana.', 'Desde mañana el equipo te ayuda con tus dudas.'], ['Se ha detectado un error en la factura.', 'Encontramos un error en la factura.'], ['Mi solicitud consiste en cambiar el horario de la cita.', 'Quiero cambiar la hora de mi cita.'], ['El informe demuestra una mejora en los indicadores.', 'El informe muestra que los números mejoraron.'], ['Le deseamos una pronta recuperación.', 'Ojalá te mejores pronto.'], ['El documento debe ser firmado por ambas partes.', 'Las dos partes tienen que firmar el documento.'], ['¿Podría facilitarme el número de contacto del responsable?', '¿Me pasas el número de la persona encargada?'], ['La sesión será grabada con fines académicos.', 'Vamos a grabar la clase para usarla después.'], ['Por razones de seguridad, utilice la salida posterior.', 'Por seguridad, sal por la puerta de atrás.'], ['Me permito presentar una sugerencia para el equipo.', 'Tengo una idea para comentarles al equipo.'], ['El presupuesto no contempla gastos de transporte.', 'En el presupuesto no pusimos la plata del transporte.'], ['Es un placer invitarle a la presentación del libro.', 'Nos hace ilusión invitarte a la presentación del libro.'], ['Se efectuará una revisión exhaustiva de los equipos.', 'Vamos a revisar bien todos los equipos.'], ['Le solicito que conserve este comprobante de pago.', 'Guarda este recibo de pago, porfa.'], ['Los resultados estarán disponibles en la plataforma virtual.', 'Puedes ver los resultados en la plataforma.'], ['La próxima semana se publicará el calendario definitivo.', 'La otra semana suben el calendario final.'], ['Lamento no haber respondido con anterioridad.', 'Perdona, se me pasó responderte antes.'], ['Es preciso considerar las limitaciones del estudio.', 'También hay que tener en cuenta lo que limita el estudio.'], ['El restaurante dispone de opciones vegetarianas.', 'En ese restaurante también hay comida vegetariana.'], ['Comunique cualquier cambio al departamento correspondiente.', 'Si cambia algo, avísale al departamento.'], ['A continuación, se presentan los principales hallazgos.', 'Aquí están las cosas más importantes que encontramos.'], ['Le agradecería que cerrara la ventana al salir.', '¿Cierras la ventana cuando salgas, porfa?'], ['Su pedido fue despachado esta mañana.', 'Tu pedido salió esta mañana.'], ['No contamos con información suficiente para tomar una decisión.', 'Todavía nos faltan datos para decidir.'], ['La universidad otorgará becas a estudiantes destacados.', 'La universidad les dará becas a quienes tengan buenas notas.'], ['Le invitamos a consultar las bases de la convocatoria.', 'Échale un vistazo a las bases de la convocatoria.'], ['El plazo de inscripción ha concluido.', 'Ya se acabó el tiempo para inscribirse.'], ['Por favor, proceda a reiniciar el dispositivo.', 'Prueba apagar y prender el aparato.'], ['Las entradas se encuentran disponibles en línea.', 'Las entradas ya se pueden comprar por internet.'], ['Agradecemos su interés en nuestra institución.', 'Gracias por interesarte en nuestra institución.'], ['El curso está dirigido a personas sin experiencia previa.', 'Este curso es para quienes recién empiezan.'], ['Su cuenta ha sido bloqueada temporalmente por seguridad.', 'Por seguridad, tu cuenta quedó bloqueada por un rato.'], ['Se ruega no interrumpir a los expositores.', 'Porfa, no interrumpan a quienes están exponiendo.'], ['Nos complace comunicarle que ha sido seleccionado.', '¡Buenas noticias! Te escogieron.'], ['Los postulantes deben cumplir todos los requisitos.', 'Para postular hay que cumplir todos los requisitos.'], ['El proveedor confirmó la recepción del pedido.', 'El proveedor ya nos dijo que recibió el pedido.'], ['Se suspenderá el suministro de agua por trabajos de reparación.', 'Van a cortar el agua porque están arreglando una tubería.'], ['Por medio de la presente, solicito acceso a los registros.', 'Necesito que me den acceso a esos registros.'], ['Los asistentes recibirán un certificado al finalizar el taller.', 'Al terminar el taller les van a dar un certificado.'], ['El pago debe realizarse antes del vencimiento.', 'No te olvides de pagar antes de que venza.'], ['Agradezco de antemano su atención a esta petición.', 'Gracias desde ya por revisar lo que pedí.'], ['El horario de atención ha sido modificado.', 'Cambiaron el horario de atención.'], ['Le recomendamos consultar a un profesional especializado.', 'Sería bueno que hables con un especialista.'], ['La mercancía presenta daños visibles en el empaque.', 'La caja del producto llegó golpeada.'], ['Los archivos adjuntos superan el tamaño permitido.', 'Los archivos pesan más de lo que se puede enviar.'], ['Su participación es fundamental para el éxito de esta iniciativa.', 'Tu ayuda es clave para que esta idea salga bien.'], ['En caso de emergencia, comuníquese con recepción.', 'Si pasa algo urgente, llama a recepción.'], ['La evaluación tendrá una duración de noventa minutos.', 'Tienen hora y media para hacer la prueba.'], ['Le informaremos cuando exista una respuesta oficial.', 'Te avisamos cuando tengamos una respuesta oficial.'], ['Se requiere autorización previa para ingresar al edificio.', 'Hay que pedir permiso antes de entrar al edificio.'], ['El equipo acordó posponer la decisión hasta mañana.', 'Quedamos en decidir eso mañana.'], ['Es importante conservar los materiales en buen estado.', 'Cuiden los materiales para que no se dañen.'], ['La reserva se encuentra confirmada para dos personas.', 'Tu reserva para dos ya está lista.'], ['Le agradecería que compartiera el enlace de acceso.', '¿Me compartes el enlace para entrar?'], ['La exposición abordará las causas del cambio climático.', 'La presentación va a tratar sobre las causas del cambio climático.'], ['La documentación deberá entregarse en formato digital.', 'Hay que entregar los documentos en digital.'], ['El personal de soporte responderá durante el horario laboral.', 'Soporte te responde en horas de trabajo.'], ['Nos disculpamos por cualquier inconveniente que esto pudiera causar.', 'Perdón si esto les causa algún problema.'], ['El proyecto dispone de recursos limitados.', 'Para el proyecto tenemos pocos recursos.'], ['Se ha programado una entrevista para el martes.', 'Te agendaron una entrevista para el martes.'], ['Le solicitamos abstenerse de compartir información confidencial.', 'No compartas información privada, por favor.'], ['La actividad requiere la formación de grupos de cuatro integrantes.', 'Para esta actividad, hagan grupos de cuatro.'], ['Quedo atento a cualquier consulta adicional.', 'Si tienes otra duda, me avisas.'], ['El plazo podrá ampliarse de acuerdo con las circunstancias.', 'Dependiendo de lo que pase, tal vez den más tiempo.'], ['Los pasajeros deberán permanecer sentados durante el despegue.', 'Durante el despegue, todos deben quedarse sentados.'], ['El museo ofrece visitas guiadas los fines de semana.', 'Los fines de semana puedes recorrer el museo con un guía.'], ['La investigación presenta resultados preliminares prometedores.', 'Los primeros resultados del estudio se ven bastante bien.'], ['Agradeceremos que notifique su ausencia con anticipación.', 'Si no vas a venir, avisa con tiempo, porfa.'], ['La aplicación requiere una conexión estable a internet.', 'Para usar la aplicación necesitas buena conexión.'], ['El contrato entrará en vigor el primer día del mes.', 'El contrato empieza a valer el primer día del mes.'], ['Se recomienda asistir con ropa cómoda.', 'Mejor ven con ropa cómoda.'], ['La compra incluye una garantía de doce meses.', 'La compra viene con un año de garantía.'], ['Le comunicaremos oportunamente cualquier novedad.', 'Te contamos si hay alguna novedad.'], ['El documento adjunto contiene las instrucciones completas.', 'En el archivo adjunto están todas las instrucciones.'], ['No fue posible procesar su solicitud en este momento.', 'Ahorita no pudimos procesar tu solicitud.'], ['El comité evaluará las propuestas recibidas.', 'El comité va a revisar las propuestas que llegaron.'], ['El pedido puede retirarse en nuestra oficina principal.', 'Puedes recoger tu pedido en la oficina principal.'], ['Le pedimos que actualice sus datos de contacto.', 'Actualiza tus datos para poder contactarte.'], ['La entrega se efectuará una vez confirmado el pago.', 'Te entregamos el producto cuando se confirme el pago.'], ['Se ha alcanzado el número máximo de participantes.', 'Ya se llenaron todos los cupos.'], ['Es conveniente revisar las instrucciones antes de comenzar.', 'Antes de empezar, lee bien las instrucciones.'], ['El conductor deberá presentar su licencia vigente.', 'El conductor tiene que llevar la licencia al día.'], ['La reunión concluyó sin alcanzar un acuerdo.', 'La reunión terminó y nadie llegó a un acuerdo.']]
registros_estilo = []
for grupo, (formal, informal) in enumerate(PARES_ESTILO, start=1):
    for label, texto in enumerate((formal, informal)):
        registros_estilo.append({"grupo": grupo, "texto": texto,
                                 "estilo": NOMBRES_ESTILO[label], "label": label})
df_estilo = pd.DataFrame(registros_estilo)
assert len(df_estilo) >= 200 and df_estilo["texto"].str.casefold().is_unique
assert df_estilo["label"].value_counts().nunique() == 1
# El CSV se guarda en el directorio de trabajo del notebook.
df_estilo.to_csv("dataset_estilos_espanol.csv", index=False, encoding="utf-8-sig")
print(f"Total: {len(df_estilo)} frases en {df_estilo.grupo.nunique()} grupos")
display(df_estilo.sample(8, random_state=SEMILLA_ESTILO))

In [ ]:
grupos = df_estilo["grupo"].unique()
gr_train, gr_temp = train_test_split(grupos, test_size=0.30, random_state=SEMILLA_ESTILO)
gr_val, gr_test = train_test_split(gr_temp, test_size=0.50, random_state=SEMILLA_ESTILO)
def tomar_grupos(gr):
    return df_estilo[df_estilo["grupo"].isin(gr)].reset_index(drop=True)
train_estilo, val_estilo, test_estilo = map(tomar_grupos, (gr_train, gr_val, gr_test))
assert set(gr_train).isdisjoint(gr_val) and set(gr_train).isdisjoint(gr_test) and set(gr_val).isdisjoint(gr_test)
resumen_particiones = pd.DataFrame({
    nombre: {"formal": int((parte.label == 0).sum()), "informal": int((parte.label == 1).sum()),
             "total": len(parte), "porcentaje": round(100 * len(parte) / len(df_estilo), 1)}
    for nombre, parte in (("Entrenamiento", train_estilo), ("Validación", val_estilo), ("Prueba", test_estilo))
}).T
print("Las proporciones pueden variar ligeramente por el redondeo de grupos completos.")
display(resumen_particiones)

## 2. Línea base: TF-IDF y regresión logística

El vocabulario y los pesos TF-IDF se aprenden únicamente con entrenamiento. El conjunto de prueba se reserva para la comparación final.

In [ ]:
linea_base_estilo = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
    LogisticRegression(max_iter=2000, random_state=SEMILLA_ESTILO)
)
linea_base_estilo.fit(train_estilo.texto, train_estilo.label)
print("Línea base entrenada con", len(train_estilo), "frases.")

## 3. Ajuste fino de BETO

Se tokeniza con el tokenizador propio de BETO y se selecciona el checkpoint con mejor F1-macro en **validación**. La cabeza de clasificación empieza con pesos nuevos: el aviso de pesos recién inicializados es normal. Las opciones están alineadas con la API de Transformers 5.x utilizada en el notebook.

In [ ]:
MODELO_ESTILO = "dccuchile/bert-base-spanish-wwm-cased"
tokenizador_estilo = AutoTokenizer.from_pretrained(MODELO_ESTILO)
def convertir_dataset_estilo(df):
    datos = Dataset.from_pandas(df[["texto", "label"]], preserve_index=False)
    return datos.map(lambda lote: tokenizador_estilo(lote["texto"], truncation=True, max_length=64),
                     batched=True, remove_columns=["texto"])
train_tok_estilo = convertir_dataset_estilo(train_estilo)
val_tok_estilo = convertir_dataset_estilo(val_estilo)
test_tok_estilo = convertir_dataset_estilo(test_estilo)
collator_estilo = DataCollatorWithPadding(tokenizer=tokenizador_estilo)
set_seed(SEMILLA_ESTILO)
modelo_estilo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_ESTILO, num_labels=2,
    id2label={i: nombre for i, nombre in enumerate(NOMBRES_ESTILO)},
    label2id={nombre: i for i, nombre in enumerate(NOMBRES_ESTILO)}
)
def metricas_estilo(eval_pred):
    predicciones = np.argmax(eval_pred.predictions, axis=-1)
    return {"accuracy": accuracy_score(eval_pred.label_ids, predicciones),
            "f1_macro": f1_score(eval_pred.label_ids, predicciones, average="macro", zero_division=0)}

In [ ]:
argumentos_estilo = TrainingArguments(
    output_dir="salida_beto_estilos", learning_rate=2e-5,
    per_device_train_batch_size=8, per_device_eval_batch_size=16,
    num_train_epochs=3, weight_decay=0.01, warmup_steps=0.1,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True, save_total_limit=2,
    logging_strategy="epoch", fp16=(DISPOSITIVO_ESTILO == "cuda"),
    report_to="none", seed=SEMILLA_ESTILO, data_seed=SEMILLA_ESTILO,
)
trainer_estilo = Trainer(
    model=modelo_estilo, args=argumentos_estilo, train_dataset=train_tok_estilo,
    eval_dataset=val_tok_estilo, data_collator=collator_estilo,
    processing_class=tokenizador_estilo, compute_metrics=metricas_estilo,
)
trainer_estilo.train()
print("Mejor F1-macro en validación:", trainer_estilo.state.best_metric)
print("Checkpoint seleccionado:", trainer_estilo.state.best_model_checkpoint)
trainer_estilo.save_model("modelo_beto_estilos")
tokenizador_estilo.save_pretrained("modelo_beto_estilos")

## 4. Curvas de aprendizaje

La pérdida de entrenamiento se registra una vez por época. Si la pérdida de entrenamiento disminuye y la de validación aumenta de manera sostenida, podría haber sobreajuste; con tan pocos datos, las curvas también pueden fluctuar.

In [ ]:
historial_estilo = pd.DataFrame(trainer_estilo.state.log_history)
perdida_ent = historial_estilo.dropna(subset=["loss"])
perdida_val = historial_estilo.dropna(subset=["eval_loss"])
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].plot(perdida_ent.epoch, perdida_ent.loss, "o-", label="Entrenamiento")
ejes[0].plot(perdida_val.epoch, perdida_val.eval_loss, "s-", label="Validación")
ejes[0].set(title="Pérdida por época", xlabel="Época", ylabel="Entropía cruzada")
ejes[0].legend()
ejes[1].plot(perdida_val.epoch, perdida_val.eval_accuracy, "o-", label="Accuracy")
ejes[1].plot(perdida_val.epoch, perdida_val.eval_f1_macro, "s-", label="F1-macro")
ejes[1].set(title="Métricas en validación", xlabel="Época", ylabel="Valor", ylim=(0, 1.05))
ejes[1].legend()
plt.tight_layout(); plt.show()
display(perdida_val[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]].round(3))

## 5. Evaluación y comparación en prueba

Ambos modelos se evalúan sobre las mismas frases que no participaron en el entrenamiento ni en la selección del checkpoint. Se presentan métricas por clase y matrices con cantidades absolutas.

In [ ]:
y_real_estilo = test_estilo.label.to_numpy()
pred_base_estilo = linea_base_estilo.predict(test_estilo.texto)
salida_beto = trainer_estilo.predict(test_tok_estilo)
pred_beto_estilo = np.argmax(salida_beto.predictions, axis=-1)
comparacion_estilo = pd.DataFrame([
    {"Modelo": nombre, "Accuracy": accuracy_score(y_real_estilo, pred),
     "F1-macro": f1_score(y_real_estilo, pred, average="macro", zero_division=0)}
    for nombre, pred in [("TF-IDF + regresión logística", pred_base_estilo), ("BETO ajustado", pred_beto_estilo)]
])
display(comparacion_estilo.round(3))
for nombre, pred in [("TF-IDF + regresión logística", pred_base_estilo), ("BETO ajustado", pred_beto_estilo)]:
    print("\n", nombre)
    print(classification_report(y_real_estilo, pred, labels=[0, 1], target_names=NOMBRES_ESTILO, digits=3, zero_division=0))
fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for eje, nombre, pred in zip(ejes, ["TF-IDF + regresión logística", "BETO ajustado"], [pred_base_estilo, pred_beto_estilo]):
    ConfusionMatrixDisplay.from_predictions(y_real_estilo, pred, labels=[0, 1],
        display_labels=NOMBRES_ESTILO, cmap="Blues", ax=eje, colorbar=False)
    eje.set_title(nombre); eje.set_xlabel("Predicción"); eje.set_ylabel("Etiqueta real")
plt.tight_layout(); plt.show()

## 6. Análisis de 10 errores

Se muestran primero errores de BETO en prueba; si no hay diez, se añaden errores distintos de TF-IDF en prueba y, si aún faltan, de TF-IDF en validación. La columna «partición» identifica su procedencia: solo se usa prueba para la comparación final. Cada comentario señala una **hipótesis**, no una explicación demostrada del modelo. Revise las frases y complemente los comentarios teniendo en cuenta contexto, ambigüedad y posibles errores de etiquetado. Si hay menos de diez errores disponibles, no se inventan fallos.

In [ ]:
def comentario_error_estilo(texto, real, predicha):
    t = texto.lower()
    pistas = []
    if any(x in t for x in ("porfa", "profe", "súper", "ahorita", "plata", "chico")):
        pistas.append("contiene vocabulario coloquial")
    if any(x in t for x in ("agradecer", "solicito", "le comunic", "se ruega", "por medio de la presente")):
        pistas.append("contiene fórmulas institucionales")
    if any(x in t for x in ("por favor", "gracias", "disculp", "perdón")):
        pistas.append("la cortesía también aparece en registros conversacionales")
    if "?" in texto or "¿" in texto:
        pistas.append("la pregunta admite distintas formas de trato")
    if not pistas:
        pistas.append("la frase puede ser neutra o depender del contexto")
    return (f"Se etiquetó como {real} y se predijo {predicha}; " + "; ".join(pistas) +
            ". Hipótesis: revisar el contexto y las expresiones aprendidas; no demuestra la causa del fallo.")
filas_errores = []
frases_revisadas = set()
# La validación solo complementa el análisis; no modifica los resultados de prueba.
fuentes_errores = [
    ("BETO", "prueba", test_estilo, pred_beto_estilo),
    ("TF-IDF", "prueba", test_estilo, pred_base_estilo),
    ("TF-IDF", "validación", val_estilo, linea_base_estilo.predict(val_estilo.texto)),
]
for nombre, particion, datos, preds in fuentes_errores:
    reales = datos.label.to_numpy()
    for i in np.flatnonzero(preds != reales):
        real, pred = NOMBRES_ESTILO[int(reales[i])], NOMBRES_ESTILO[int(preds[i])]
        texto = datos.iloc[i].texto
        if texto in frases_revisadas:
            continue
        frases_revisadas.add(texto)
        filas_errores.append({"modelo": nombre, "partición": particion, "frase": texto,
                              "real": real, "predicha": pred,
                              "análisis preliminar": comentario_error_estilo(texto, real, pred)})
errores_estilo = pd.DataFrame(filas_errores, columns=["modelo", "partición", "frase", "real", "predicha", "análisis preliminar"])
print(f"Errores distintos disponibles: {len(errores_estilo)}; se muestran hasta 10.")
if len(errores_estilo) < 10:
    print("Hay menos de 10 errores reales; amplíe el dataset antes de exigir diez análisis.")
with pd.option_context("display.max_colwidth", None):
    display(errores_estilo.head(10))

## 7. Conclusión basada en los resultados

La siguiente celda redacta una síntesis a partir de las métricas calculadas. Antes de entregar, complemente la interpretación con sus observaciones de las curvas y de los errores concretos. Recuerde que la comparación se limita a este conjunto sintético y a una sola partición.

In [ ]:
f1_base = float(comparacion_estilo.loc[comparacion_estilo.Modelo == "TF-IDF + regresión logística", "F1-macro"].iloc[0])
f1_beto = float(comparacion_estilo.loc[comparacion_estilo.Modelo == "BETO ajustado", "F1-macro"].iloc[0])
diferencia = f1_beto - f1_base
if abs(diferencia) < 1e-9:
    comparacion_texto = "ambos modelos obtuvieron el mismo F1-macro"
elif diferencia > 0:
    comparacion_texto = f"BETO superó a TF-IDF en {diferencia:.3f} puntos de F1-macro"
else:
    comparacion_texto = f"TF-IDF superó a BETO en {-diferencia:.3f} puntos de F1-macro"
print(f"En el conjunto de prueba, {comparacion_texto}. "
      f"BETO obtuvo {f1_beto:.3f} y la línea base {f1_base:.3f}. "
      "El ajuste fino permite adaptar un modelo preentrenado al registro del texto, "
      "pero debe valorarse junto al costo de entrenamiento y frente a una línea base sencilla. "
      "Como las frases son sintéticas, el conjunto es pequeño y solo se utilizó una partición, "
      "se necesitan datos reales y evaluaciones adicionales antes de generalizar estos resultados.")